In [9]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['proyecto_bigdata']
coleccion = db['datos_scraping'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")


Conexión exitosa a MongoDB


In [10]:
import time
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# ================= CONFIG =================
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)

NOMBRE = "Javiera Pizarro"
datos_finales = []
LIMITE_DATOS = 150

# ================= LISTADO =================
url = "https://prd-next.autopia.cl/"
driver.get(url)
time.sleep(8)

# ================= SCROLL MEJORADO =================
last_height = driver.execute_script("return document.body.scrollHeight")

for i in range(50):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)

    new_height = driver.execute_script("return document.body.scrollHeight")
    print(f"Scroll {i+1} - altura: {new_height}")

    if new_height == last_height:
        print("⚠️ No cargan más autos")
        break

    last_height = new_height

time.sleep(5)

# ================= SACAR LINKS =================
autos = driver.find_elements(By.TAG_NAME, "a")

links = []

for a in autos:
    href = a.get_attribute("href")
    if href and "prd-next.autopia.cl/auto-usado" in href and "wa.me" not in href:
        links.append(href)

# Quitar duplicados manteniendo orden
links = list(dict.fromkeys(links))

# Limitar a 150
links = links[:LIMITE_DATOS]

print("Total links únicos encontrados:", len(links))

# ================= DETALLE =================
for link in links:
    try:
        driver.get(link)
        time.sleep(4)

        texto = driver.find_element(By.TAG_NAME, "body").text

        # PRECIO
        precio_match = re.search(r"\$\s?[\d\.]+", texto)
        precio = precio_match.group(0) if precio_match else None

        # MARCA Y MODELO DESDE URL
        try:
            partes_url = link.split("/auto-usado/")[1].split("/")
            marca = partes_url[0].replace("%20", " ").upper()
            modelo = partes_url[1].replace("%20", " ").upper()
        except:
            marca = None
            modelo = None

        # CARACTERÍSTICAS
        caracteristicas = {}
        items = driver.find_elements(By.CSS_SELECTOR, "ul.mfpdp-list-ul li")

        for item in items:
            texto_item = item.text.strip()
            if ":" in texto_item:
                clave, valor = texto_item.split(":", 1)
                caracteristicas[clave.strip().lower()] = valor.strip()

        anio = caracteristicas.get("año")
        kilometraje = caracteristicas.get("kilometraje")
        combustible = caracteristicas.get("combustible")

        # CIUDAD
        try:
            direccion = driver.find_element(By.CSS_SELECTOR, "p.mfpdp-cl-direction").text.strip()
            ciudad = direccion.split("-")[-1].strip() if "-" in direccion else direccion
        except:
            ciudad = "No disponible"

        dato = {
            "marca": marca,
            "modelo": modelo,
            "anio": anio,
            "kilometraje": kilometraje,
            "combustible": combustible,
            "ciudad": ciudad,
            "url": link,
            "precio": precio,
            "nombre": NOMBRE
        }

        datos_finales.append(dato)

        print("OK:", marca, modelo, ciudad)

    except Exception as e:
        print("Error:", link, e)

driver.quit()

# ================= MONGO SIN DUPLICADOS =================
client = MongoClient("mongodb://mongodb:27017/")
db = client["proyecto_bigdata"]
coleccion = db["datos_scraping"]

procesados = 0
nuevos = 0
actualizados = 0

for dato in datos_finales:
    resultado = coleccion.update_one(
        {"url": dato["url"]},
        {"$set": dato},
        upsert=True
    )

    procesados += 1

    if resultado.upserted_id is not None:
        nuevos += 1
    else:
        actualizados += 1

print("🔥 PROCESO TERMINADO")
print("📌 Datos procesados:", procesados)
print("✅ Nuevos guardados:", nuevos)
print("♻️ Repetidos actualizados:", actualizados)
print("📦 Total en Mongo:", coleccion.count_documents({}))

Scroll 1 - altura: 3697
⚠️ No cargan más autos
Total links únicos encontrados: 20
OK: RENAULT KOLEOS Santiago
OK: HAVAL H2 Santiago
OK: GREAT WALL POER Santiago
OK: JAC T6 Santiago
OK: DONGFENG 500 Santiago
OK: DONGFENG 500 Santiago
OK: DONGFENG 500 Santiago
OK: DONGFENG 560 Santiago
OK: HONDA ZR-V Santiago
OK: SUBARU EVOLTIS Santiago
OK: MAZDA CX-60 Santiago
OK: SUZUKI VITARA Santiago
OK: DONGFENG SUV 560 Santiago
OK: CHANGAN CS15 Santiago
OK: JETOUR DASHING Santiago
OK: MG HS Santiago
OK: PEUGEOT 2008 Santiago
OK: MG HS Santiago
OK: HAVAL JOLION Santiago
OK: SUZUKI JIMNY Santiago
🔥 PROCESO TERMINADO
📌 Datos procesados: 20
✅ Nuevos guardados: 20
♻️ Repetidos actualizados: 0
📦 Total en Mongo: 20
